In [1]:
import numpy as np
from scipy.integrate import odeint
from pysindy.feature_library import PolynomialLibrary
import matplotlib.pyplot as plt
import pysindy as ps
from scipy.optimize import fsolve
from pysindy.differentiation import FiniteDifference
import pandas
from mpl_toolkits import mplot3d
from scipy.optimize import curve_fit
from pysindy.feature_library import CustomLibrary
import math
import pandas as pd
import torch
import torch.optim as optim
import time


In [2]:
#!pip install pysindy # I needed this to use colab. You do not need to run this

In [3]:
#!pip install scipy==1.10.1 #needed for colab. You do not need to run

In [4]:
#!pip install --force-reinstall numpy==1.23.5 #needed for colab. You do not need to run

In [5]:
#original logistic growth equation with control
def logistic_growth_controlled(y, t, params, treatment_function, cycles):
    """
    Differential equation for logistic growth with a control term for Leukemic stem cells in active

    Args:
        y (float): Current population size.
        t (float): Current time.
        params (array float): array with parameter values.
        treatment_function : A function that takes time and cycles (t, cycle) and returns  u in {0,1}  ( 0 means no treatment and 1 treatment).
        cycles : treatment cycles

    Returns:
        float: The rate of change of the population size (dy/dt).
    """
    control = treatment_function(t,cycles)
    p_l, K_A, d_k, c = params
    dydt = ( p_l * (1 - y / K_A) - c*control -d_k ) * y
    return dydt

In [6]:
#combinations of polynomial basis functions up to order 2
def recovered_model(y, t, params, treatment_function, cycles):
    """
    Differential equation for model recovered by SINDy

    Args:
        y (float): Current population size.
        t (float): Current time.
        params (array float): array with parameter values.


    Returns:
        float: The rate of change of the population size (dy/dt).
    """
    control = treatment_function(t,cycles)
    a0,a1,a2,a3,a4=params
    dydt = a0*y+a1*control+a2*y*y+a3*control*control+a4*y*control
    return dydt

In [7]:
from google.colab import drive #this was for colab as well; you do not need
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [8]:
## This is how we read the treatment data for a patient
def treatment_cycle( patient_id ) :

    #patient_id = 104
    # read all data.  [[ Boris ]] Its inefficient to read all data here; but we can optimize this later
    chemo_data = pd.read_csv('gdrive/My Drive/rsif20200091_si_003.csv')
    # get treatment information from a specific patient
    chemo_row = chemo_data[chemo_data.PatientID==patient_id].iloc[0]

    chemo_start_times = [int(x) for x in chemo_row['Chemo_begin'].split()] # starting times of therapy cycle
    chemo_end_times = [int(x) for x in chemo_row['Chemo_end'].split()] #

    return   [ chemo_start_times , chemo_end_times ]    ## [[ Boris ]] we may need a better data structure here


In [9]:
# Define a control function (example: constant control after a certain time)
def treatment_function(time, cycles ):

    chemo_start_times, chemo_end_times  = cycles

    Ncycles = len(chemo_start_times)

    for i in range(Ncycles) :
        if chemo_start_times[i] <= time <= chemo_end_times[i]  :
            return 1.0

    return 0.0  # No control


In [10]:
# Setting parameters for simulations

p_l = 0.11337807
K_A = 10000
d_k = 1/30
c = 1.2114013

# Initial value

t = np.linspace(0,212,20000) # Time points
num_init_cond = 1
y_init_values=[1000]

delta_t=t[1]-t[0] #time step
print(delta_t)

print(y_init_values)
#c_values= np.random.uniform(1.10,1.30,10) #try various values of c
#print(c_values)

c_values=[1.14780485, 1.17477361, 1.27668946, 1.12613127, 1.10723674, 1.27374855,
 1.10596159, 1.14461985, 1.18061548, 1.26345782]






0.010600530026501326
[1000]


In [11]:
#We use the Adam algorithm to personalize the c parameter
#We first use SINDy to find an initial guess for c_common

#Generate 10 trajectories for the same patient by changing the c-value
patient_id = 104
cycles = treatment_cycle(patient_id)
params = [p_l, K_A, d_k, c]
recovered_c_values=[]
control_values=[]
MSE_values=[]
RSE_values=[]

for time in t:
          control_values.append(treatment_function(time,cycles)) #get the control values for each time point

#Generate 10 trajectories for the same patient by changing the c-value
#Use Sindy to estimate the c-value for each trajectory
#Take the average of the c-values recovered from SINDy; this will be our initial guess for the common component
for value in c_values:
      params[3]=value #change c for each traj.
      La_o=y_init_values[0]
      results = odeint(logistic_growth_controlled, La_o , t, args=(params, treatment_function, cycles))
      results=np.array(results)
      control_values=np.array(control_values)
      data=results #only the y_values
      data_c=np.c_[results,control_values] #adds control_values as a second column to do SINDY w control
      print(control_values.shape)
      print(data_c.shape)

      fd=FiniteDifference() #take the derivative of each trajectory
      data_prime=fd._differentiate(data,t) #derivative is just of the y-values (does not include the c_values)
      print("The size of X' is ", data_prime.shape)

      #Create a library of basis functions (of y and control) for sindy to use
      theta_lib_c=PolynomialLibrary(degree=2) #restrict to polynomials of deg 2
      functions = [lambda x : x, lambda x : x*x, lambda x ,y : x*y] #restrict only to the fcns that show up
      customlib=CustomLibrary(library_functions=functions)
      customlib.fit(data_c) #fit to our data, including the control
      print(customlib.get_feature_names())
      theta_c=customlib.transform(data_c) #transform to the correct size (,5)
      #print("The size of theta_c is", theta_c.shape)


      #Find the coefficents of the basis fcns
      #solve X'=Theta(X)*Xi for Xi
      #use least squares solution as initial guess
      Xi=np.linalg.lstsq(theta_c,data_prime)[0] #Is our initial guess using Python's least squares solver
      print("Xi is ", Xi)

      Xi_row,Xi_col=Xi.shape

      Lambda=0.00000001 #Sparsification knob (this was the value given in the PNAS paper for Lorenz)
            #This loop is adapted from the Sindy_PNAS paper, SI Appendix and converted into Python
      for k in range(1000):
        smallinds=(np.absolute(Xi)<Lambda) #find all "very small" coefficients
               #print(smallinds)
        Xi[smallinds]=0 #set them=0
            #print(Eps_P)
        for ind in range(Xi_col): #Xi_col is the number of state variables
           biginds=np.logical_not(smallinds[:,ind]) #regress dynamics onto remaining terms
           Xi[biginds,ind]=np.linalg.lstsq(theta_c[:,biginds],data_prime[:,ind])[0]
      print(Xi)
      print(Xi.shape)

      a0=Xi[0][0]
      a1=Xi[1][0]
      a2=Xi[2][0]
      a3=Xi[3][0]
      a4=Xi[4][0] #the recovered value of c

      recovered_c_value=-a4 #the general model recovered by SINDy is y'=a0*y+a1*control+a2*y^2+a3*control^2+a4*y*control
      #When simplified, the Hoffman model is y'=(p_l-D_k)*y-(p_l/K_A)*y^2-c*y*control. Thus, a0=p_l-d_k, a1=0, a_2=(-p_l/K_A), a3=0, and a4=-c
      #so c=-a4

      recovered_c_values.append(recovered_c_value)
      print("The recovered c value is ", recovered_c_value)

      #generate a new trajectory using the c-value recovered from SINDy
      params_recovered=[a0,a1,a2,a3,a4]
      results_hat = odeint(recovered_model, La_o, t, args=(params_recovered,treatment_function,cycles))

      #Find the MSE and RSE from SINDy
      MSE_sindy = np.square(np.subtract(results[:,0],results_hat[:,0])).mean()
      MSE_values.append(MSE_sindy)

      RSE_sindy =  MSE_sindy/np.linalg.norm(results[:,0])
      RSE_values.append(RSE_sindy)


print("The average MSE from SINDy is", np.average(MSE_values))
print("The average relative squared error is", np.average(RSE_values))

#our initial guess for the common component
c_common=np.average(recovered_c_values)
print("The recovered common component is", c_common)
print("The recovered c values are", recovered_c_values)

#the average of all the c-values
true_c_common=np.average(c_values)
print("The actual common component is", true_c_common)
print("The actual c values in order are", c_values)


print(true_c_common-c_common)






















(20000,)
(20000, 2)
The size of X' is  (20000, 1)
['f0(x0)', 'f0(x1)', 'f1(x0)', 'f1(x1)', 'f2(x0,x1)']
Xi is  [[ 7.93386572e-02]
 [ 5.15012193e-04]
 [-1.13255304e-05]
 [ 5.15012193e-04]
 [-1.14713126e+00]]


<__array_function__ internals>:180: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.


[[ 7.93386572e-02]
 [ 5.15012193e-04]
 [-1.13255304e-05]
 [ 5.15012193e-04]
 [-1.14713126e+00]]
(5, 1)
The recovered c value is  1.1471312553329145
(20000,)
(20000, 2)
The size of X' is  (20000, 1)
['f0(x0)', 'f0(x1)', 'f1(x0)', 'f1(x1)', 'f2(x0,x1)']
Xi is  [[ 7.93217701e-02]
 [ 4.45653714e-04]
 [-1.13251129e-05]
 [ 4.45653714e-04]
 [-1.17408483e+00]]
[[ 7.93217701e-02]
 [ 4.45653714e-04]
 [-1.13251129e-05]
 [ 4.45653714e-04]
 [-1.17408483e+00]]
(5, 1)
The recovered c value is  1.1740848271333137
(20000,)
(20000, 2)
The size of X' is  (20000, 1)
['f0(x0)', 'f0(x1)', 'f1(x0)', 'f1(x1)', 'f2(x0,x1)']
Xi is  [[ 7.92586902e-02]
 [ 2.67039312e-04]
 [-1.13224623e-05]
 [ 2.67039312e-04]
 [-1.27594573e+00]]
[[ 7.92586902e-02]
 [ 2.67039312e-04]
 [-1.13224623e-05]
 [ 2.67039312e-04]
 [-1.27594573e+00]]
(5, 1)
The recovered c value is  1.2759457331648052
(20000,)
(20000, 2)
The size of X' is  (20000, 1)
['f0(x0)', 'f0(x1)', 'f1(x0)', 'f1(x1)', 'f2(x0,x1)']
Xi is  [[ 7.93523552e-02]
 [ 5.8032840

In [12]:
#the initial guess for the parameters
#we will update the parameter values using Adam
#We initialize the common components as the average c_value recovered by SINDy
#Initialize the personalize components as zero

#init_guess_common=[0] #initialize the common component as 0 (initializing everything as 0)
init_guess_common=[c_common] #initialize the common component as the avg from SINDy

#As there is a personalized component for each trajectory, the number of personalized components is the number of training trajectories; i.e. len(c_values)
init_guess_pers=np.zeros(len(c_values))

init_guess=np.concatenate((init_guess_common,init_guess_pers)) #put common and personalized components in a single vector
init_guess=np.c_[init_guess] #make the correct shape

print(init_guess)

init_guess=torch.tensor(init_guess, requires_grad=True) #convert to a tensor to use PyTorch
init_guess_common=torch.tensor(init_guess_common, requires_grad=True)
init_guess_pers=torch.tensor(init_guess_pers, requires_grad=True)


[[1.179413]
 [0.      ]
 [0.      ]
 [0.      ]
 [0.      ]
 [0.      ]
 [0.      ]
 [0.      ]
 [0.      ]
 [0.      ]
 [0.      ]]


In [13]:
#Our loss function to be optimized
#Takes in a tensor of parameter values as input
#Returns sum_{i=1}^{n} ||X_dot-Theta(X)(Eps_c+Eps_p)||_2^2+||Eps_c+Eps_p_i||_1+||Eps_p_i||_2 (the loss function)
def loss(init_guess):
  sum=0 #initialize sum
  for i in range(len(c_values)): #loop through each c-value to generate a trajectory
            #generate the data for each trajectory
            #For each trajectory we only change the c-value

            La_o=y_init_values[0] #init condition
            params=[p_l,K_A,d_k,c_values[i]] #original parameters; Fix p_l,K_A,d_k; vary c
            results=odeint(logistic_growth_controlled,La_o,t,args=(params,treatment_function,cycles))
            results= np.array(results)
            W=np.c_[results,control_values]

            #separate the data into y-values and control values
            y_values=W[:,0] #y values eval at time points
            controls=W[:,1] #control values eval at time points
            fd=FiniteDifference() #take the derivative of each trajectory

            y_values_prime=fd._differentiate(y_values,t) #derivative of y_values




            #convert into tensors
            y_values_tensor = torch.tensor(y_values, dtype=init_guess.dtype)
            controls_tensor = torch.tensor(controls, dtype=init_guess.dtype)
            y_values_prime_tensor = torch.tensor(y_values_prime, dtype=init_guess.dtype)


            #Estimate the derivatives
            #W_prime is the true derivative (the right hand side of the original ODE with the true parameter values)
            #W_prime_hat is the estimated derivative (the right hand side of ODE, with c=init_guess_commom+init_guess_pers_i)

            #The true coefficient of every basis, for example, (pl-dk) is the original coefficient of y in the ODE
            pars=[(p_l-d_k), 0, -p_l/K_A, 0, -c_values[i]]

            W_prime=y_values_prime_tensor
            W_prime_hat = pars[0]*y_values_tensor + pars[1]*controls_tensor+ (pars[2])*y_values_tensor**2 + pars[3]*controls_tensor**2-(init_guess[0]+init_guess[i+1])*y_values_tensor*controls_tensor

            #parameters for the loss function
            Lambda_1=0.01
            Lambda_2=0.01



            #Take the loss function for each trajectory and then sum them up
            sum+=torch.linalg.norm(W_prime-W_prime_hat)**2 + Lambda_1*torch.linalg.norm(torch.tensor([init_guess[0],init_guess[i+1]]), ord=1) + Lambda_2*(torch.linalg.norm(torch.tensor([init_guess[i+1]])))**2

  return sum

In [14]:
print(loss(init_guess))

tensor(1729538.0672, dtype=torch.float64, grad_fn=<AddBackward0>)


In [15]:
#Run the Adam algorithm to simulataneously recover the common and personalized components for training

init_guess=torch.tensor(init_guess, requires_grad=True)
optimizer=optim.Adam([init_guess],lr=0.001) #creates an Adam Optimizer object, lr is learning rate, use the default value from PyTorch
prev=0 #represent the loss value of the previous iteration
step=0 #the iteration number
loss_value=100 #use to represent the current loss value; I initialized as a random value just so loss_value and prev wouldn't start out the same

while(abs(loss_value-prev)>0.0001): #until convergence (algorithm terminates when current and previous loss function are within 0.0001 of each other)
  prev=loss_value #set prev=the loss value of the previous iteration
  optimizer.zero_grad() #reset the gradient
  loss_value=loss(init_guess) #take loss value of current iteration
  loss_value.backward() #take the gradient
  optimizer.step() #update parameters using Adam, step forward
  print("This is step", step, "and the loss is", loss_value.item())
  step+=1

This is step 0 and the loss is 1729538.0672378207
This is step 1 and the loss is 1681523.4066026665
This is step 2 and the loss is 1635712.7171115498
This is step 3 and the loss is 1591879.7288555824
This is step 4 and the loss is 1549426.5688224456
This is step 5 and the loss is 1507672.2301681323
This is step 6 and the loss is 1466313.282151233
This is step 7 and the loss is 1425344.1617903155
This is step 8 and the loss is 1384854.7282910899
This is step 9 and the loss is 1344966.1221616906
This is step 10 and the loss is 1305783.4302898934
This is step 11 and the loss is 1267369.7050412798
This is step 12 and the loss is 1229742.814123021
This is step 13 and the loss is 1192887.59786346
This is step 14 and the loss is 1156775.810260192
This is step 15 and the loss is 1121383.2763459885
This is step 16 and the loss is 1086698.7301137191
This is step 17 and the loss is 1052724.8933604301
This is step 18 and the loss is 1019473.8885125124
This is step 19 and the loss is 986959.6181058

In [16]:
print(init_guess) #the parameter values after Adam is applied

tensor([[ 1.1945],
        [-0.0466],
        [-0.0197],
        [ 0.0822],
        [-0.0683],
        [-0.0872],
        [ 0.0793],
        [-0.0885],
        [-0.0498],
        [-0.0138],
        [ 0.0690]], dtype=torch.float64, requires_grad=True)


In [17]:
init_guess=init_guess.detach().numpy() #turn init_guess back into array to compute MSE
print(init_guess)

[[ 1.19447588]
 [-0.04664888]
 [-0.01967837]
 [ 0.08224479]
 [-0.06832409]
 [-0.08722098]
 [ 0.07930755]
 [-0.08849675]
 [-0.04983407]
 [-0.01383613]
 [ 0.06901252]]


In [18]:
#Recover the MSE and RSE of the training phase
#Generate a trajectory using the true c_value, and a trajectory using the c_value estimated by Adam
#C_estimated=estimated c_c+ estimated c_{p_i}

#initialize empty arrays to store personalized components, MSE_values, and RSE_values
pers_c_true_array=np.array([])
pers_c_difference_array=np.array([])
MSE_values=np.array([])
RSE_values=np.array([])

recovered_common=init_guess[0] #the recovered common component for all trajectories
recovered_pers=init_guess[1:]  #the recovered personalized components for each trajectory

for i in range(len(c_values)):
            c_est=recovered_common+recovered_pers[i] #the estimated value for each c_i; c_est_i=c_common_est+c_recovered_est
            #a0=p_l-d_k, a1=0, a_2=(-p_l/K_A), a3=0, and a4=-c
            params=[p_l-d_k,0,(-p_l/K_A),0,-c_values[i]] #true parameters
            params2=[p_l-d_k,0,(-p_l/K_A),0,-c_est] #parameters w estimated c_value
            La_o=y_init_values[0] #initial condition

            results_i=odeint(recovered_model,La_o,t,args=(params,treatment_function,cycles)) #true trajectory
            results_hat_i=odeint(recovered_model,La_o,t,args=(params2,treatment_function,cycles)) #trajectory with estimated value for c

            MSE = np.square(np.subtract(results_i,results_hat_i)).mean() #Take the Mean-Squared-Error of the two trajectories
            RSE=MSE/np.linalg.norm(results_i[:,0]) #MSE divided by the 2-norm of the true value


            MSE_values=np.append(MSE_values,MSE)
            RSE_values=np.append(RSE_values,RSE)

            pers_c_true=c_values[i]-true_c_common #the "true" personalized component c_(p_i) (c_p_i=c_i-c_c)
            pers_c_true_array=np.append(pers_c_true_array,pers_c_true) #add to an array

            pers_c_true_difference=pers_c_true-recovered_pers[i] #the difference between the true and recovered personalized components
            pers_c_difference_array=np.append(pers_c_difference_array,pers_c_true_difference)

print("The true common component is ", true_c_common)
print("The recovered common component is ", recovered_common)
print("Difference in common components", true_c_common-recovered_common)

print("True personalized comp", pers_c_true_array)
print("Est personalized comp", recovered_pers)
print("Difference in personalized components", pers_c_difference_array)



MSE_average=np.average(MSE_values) #find the average MSE of all trajectories
RSE_average=np.average(RSE_values) #find the average RSE for all trajectories
print("The average MSE value is " , MSE_average)
print("The average RSE value is " , RSE_average)





The true common component is  1.180103922
The recovered common component is  [1.19447588]
Difference in common components [-0.01437195]
True personalized comp [-0.03229907 -0.00533031  0.09658554 -0.05397265 -0.07286718  0.09364463
 -0.07414233 -0.03548407  0.00051156  0.0833539 ]
Est personalized comp [[-0.04664888]
 [-0.01967837]
 [ 0.08224479]
 [-0.06832409]
 [-0.08722098]
 [ 0.07930755]
 [-0.08849675]
 [-0.04983407]
 [-0.01383613]
 [ 0.06901252]]
Difference in personalized components [0.01434981 0.01434806 0.01434074 0.01435144 0.01435379 0.01433708
 0.01435442 0.01435    0.01434768 0.01434138]
The average MSE value is  5.273663892473659e-07
The average RSE value is  8.088781361313323e-11


In [19]:
#Adaptation phase
#Generate new trajectories using different c_value
#We assume the common component of each traj. is the one from training, then we personalize each trajectory

c_values_adapt= np.random.uniform(1.10,1.30,20)

#first set of c_values used for adaptation; fixed for all simulations
#c_values_adapt=[1.21469163,1.1709559,1.24880952,1.2152118,1.11996975,1.1647565,
 #1.10925619, 1.15028617, 1.16462289, 1.24691801]


#second set of c_values used for adaptation
c_values_adapt=[1.16462416,1.12966788,1.2458747,1.29709912,1.22161367,1.24629704,
 1.2801322,  1.14502666, 1.2088311,  1.28462832, 1.2863123,  1.18372755,
 1.28240245, 1.1422735,  1.26404721, 1.28704488, 1.17262492, 1.28324867,
 1.28171646, 1.19218603]

print(c_values_adapt)
print(recovered_common)
recovered_common=recovered_common[0]
print(recovered_common)

[1.16462416, 1.12966788, 1.2458747, 1.29709912, 1.22161367, 1.24629704, 1.2801322, 1.14502666, 1.2088311, 1.28462832, 1.2863123, 1.18372755, 1.28240245, 1.1422735, 1.26404721, 1.28704488, 1.17262492, 1.28324867, 1.28171646, 1.19218603]
[1.19447588]
1.1944758758808491


In [20]:
#Modify the loss function to be only in terms of the personalized components; the common components are constant (set equal to the values recovered in training)
#Almost the exact same function as before; however this takes in only the pers components as variables instead of taking both common and pers
def loss_adaptation(init_guess_pers):
  sum=0 #initialize sum
  for i in range(len(c_values_adapt)): #loop through each c-value to generate a trajectory
            #generate the data for each trajectory
            #For each trajectory we only change the c-value

            La_o=y_init_values[0] #init condition
            params=[p_l,K_A,d_k,c_values_adapt[i]] #original parameters; Fix p_l,K_A,d_k; vary c
            results=odeint(logistic_growth_controlled,La_o,t,args=(params,treatment_function,cycles))
            results= np.array(results)
            W=np.c_[results,control_values]

            #separate the data into y-values and control values
            y_values=W[:,0] #y values eval at time points
            controls=W[:,1] #control values eval at time points
            fd=FiniteDifference() #take the derivative of each trajectory

            y_values_prime=fd._differentiate(y_values,t) #derivative of y_values




            #convert into tensors
            y_values_tensor = torch.tensor(y_values, dtype=init_guess_pers.dtype)
            controls_tensor = torch.tensor(controls, dtype=init_guess_pers.dtype)

            y_values_prime_tensor = torch.tensor(y_values_prime, dtype=init_guess_pers.dtype)


            #Estimate the derivatives
            #W_prime is the true derivative (the right hand side of the original ODE with the true parameter values)
            #W_prime_hat is the estimated derivative (the right hand side of ODE, with c=init_guess_commom+init_guess_pers_i)

            #The true coefficient of every basis, for example, (pl-dk) is the original coefficient of y in the ODE
            pars=[(p_l-d_k), 0, -p_l/K_A, 0, -c_values_adapt[i]]

            W_prime=y_values_prime_tensor
            #almost EXACTLY the same as before, but c_common is a constant (the common comp from training)
            W_prime_hat = pars[0]*y_values_tensor + pars[1]*controls_tensor+ (pars[2])*y_values_tensor**2 + pars[3]*controls_tensor**2-(recovered_common+init_guess_pers[i])*y_values_tensor*controls_tensor

            #hyperparameters; can adjust later on
            Lambda_1=0.01
            Lambda_2=0.01

            #Take the loss function for each trajectory and then sum them up
            sum+=torch.linalg.norm(W_prime-W_prime_hat)**2 + Lambda_1*torch.linalg.norm(torch.tensor([recovered_common,init_guess_pers[i]]), ord=1) + Lambda_2*(torch.linalg.norm(torch.tensor([init_guess_pers[i]])))**2

  return sum

In [21]:
#There's a personalized component for each trajectory in adaptation
init_guess_pers=np.zeros(len(c_values_adapt))
init_guess_pers=torch.tensor(init_guess_pers,requires_grad=True)
print(init_guess_pers)

print(loss_adaptation(init_guess_pers))

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       dtype=torch.float64, requires_grad=True)
tensor(3544100.2295, dtype=torch.float64, grad_fn=<AddBackward0>)


In [22]:
#Use Adam to personalize each trajectory in the Adaptation

init_guess_pers=torch.tensor(init_guess_pers, requires_grad=True)
optimizer=optim.Adam([init_guess_pers],lr=0.001) #creates an Adam Optimizer object, lr is learning rate, use the default value from PyTorch
prev=0 #represent the loss value of the previous iteration
step=0 #the iteration number
loss_value=100 #use to represent the current loss value; I initialized as a random value just so loss_value and prev wouldn't start out the same

while(abs(loss_value-prev)>0.0001): #until convergence (algorithm terminates when current and previous loss function are within 0.0001 of each other)
  prev=loss_value #set prev=the loss value of the previous iteration
  optimizer.zero_grad() #reset the gradient
  loss_value=loss_adaptation(init_guess_pers) #take loss value of current iteration
  loss_value.backward() #take the gradient
  optimizer.step() #update parameters using Adam, step forward
  print("This is step", step, "and the loss is", loss_value.item())
  step+=1

This is step 0 and the loss is 3544100.22946338
This is step 1 and the loss is 3449696.7407188313
This is step 2 and the loss is 3356992.615600573
This is step 3 and the loss is 3265997.3160885926
This is step 4 and the loss is 3176693.7392162327
This is step 5 and the loss is 3089060.5787476175
This is step 6 and the loss is 3003101.8979470227
This is step 7 and the loss is 2918839.498915528
This is step 8 and the loss is 2836295.8989486573
This is step 9 and the loss is 2755486.072749907
This is step 10 and the loss is 2676414.4466447732
This is step 11 and the loss is 2599074.3150675846
This is step 12 and the loss is 2523449.7385850316
This is step 13 and the loss is 2449519.827301367
This is step 14 and the loss is 2377263.1577012697
This is step 15 and the loss is 2306659.8411319517
This is step 16 and the loss is 2237691.1822626125
This is step 17 and the loss is 2170338.3748042695
This is step 18 and the loss is 2104581.3253121995
This is step 19 and the loss is 2040398.0717468

In [23]:
recovered_common=torch.tensor([recovered_common])
print(recovered_common)
print(init_guess_pers)

#put the common and personalized comp back into one array
final_params=torch.cat((recovered_common,init_guess_pers))
final_params=final_params.detach().numpy() #turn init_guess back into array
print(final_params)

tensor([1.1945], dtype=torch.float64)
tensor([-0.0298, -0.0648,  0.0514,  0.1027,  0.0272,  0.0518,  0.0857, -0.0494,
         0.0144,  0.0902,  0.0919, -0.0107,  0.0880, -0.0522,  0.0696,  0.0926,
        -0.0218,  0.0888,  0.0873, -0.0023], dtype=torch.float64,
       requires_grad=True)
[ 1.19447588 -0.02982846 -0.06478689  0.05142754  0.10265206  0.02716481
  0.05184991  0.08568758 -0.04942718  0.01438136  0.09018436  0.09186859
 -0.01072385  0.08795814 -0.05218051  0.06960138  0.09260127 -0.02182719
  0.08880449  0.08727205 -0.00226482]


In [24]:
#Find the MSE of adaptation

#Arrays to contain the personalized components, MSE values, and RSE values
pers_c_true_array=[]
pers_c_array=[]
MSE_values=[]
RSE_values=[]
pers_c_difference_array=[]
count=0

for value in c_values_adapt:
            La_o=y_init_values[0]
            params=[p_l,K_A,d_k,value]
            results=odeint(logistic_growth_controlled,La_o,t,args=(params,treatment_function,cycles)) #generate trajectories using the true c_values
            results= np.array(results)

#calculate the estimated c_value. Each parameter is the estimated common + est. pers.
            c_est=final_params[0]+final_params[count+1] #c_est=recovered_common+recovered_personalized_i

            pars=[(p_l-d_k), 0, (-p_l/K_A), 0, -c_est]
            results_hat=odeint(recovered_model,La_o,t,args=(pars,treatment_function,cycles)) #trajectory generated using the estimated c_value

            #Find the MSE and RSE
            MSE = np.square(np.subtract(results,results_hat)).mean()
            RSE=MSE/np.linalg.norm(results[:,0])

            print("The MSE for trajectory ", count, "is", MSE)
            print("The RSE for trajectory ", count, "is", RSE)
            print()

            MSE_values=np.append(MSE_values,MSE)
            RSE_values=np.append(RSE_values,RSE)
            count+=1

MSE_average=np.average(MSE_values)
RSE_average=np.average(RSE_values)
print("The average MSE value is " , MSE_average)
print("The average RSE value is " , RSE_average)

The MSE for trajectory  0 is 4.993423942873987e-07
The RSE for trajectory  0 is 7.553659088002233e-11

The MSE for trajectory  1 is 4.5700810669680504e-07
The RSE for trajectory  1 is 6.802616774670589e-11

The MSE for trajectory  2 is 6.07629288970386e-07
The RSE for trajectory  2 is 9.524066858811766e-11

The MSE for trajectory  3 is 5.364632872986367e-07
The RSE for trajectory  3 is 8.588027182996545e-11

The MSE for trajectory  4 is 5.738999372197381e-07
The RSE for trajectory  4 is 8.90292292224265e-11

The MSE for trajectory  5 is 6.082491626105981e-07
The RSE for trajectory  5 is 9.535479220117073e-11

The MSE for trajectory  6 is 6.586451146445631e-07
The RSE for trajectory  6 is 1.0471571192781431e-10

The MSE for trajectory  7 is 4.753161650661311e-07
The RSE for trajectory  7 is 7.125949415812537e-11

The MSE for trajectory  8 is 5.566295795785937e-07
The RSE for trajectory  8 is 8.58734797146346e-11

The MSE for trajectory  9 is 6.789669996336234e-07
The RSE for trajectory 

In [25]:

# #adaptation step

# # create 10 new trajectories with common component= the one we found
# # find the personalized component and then the MSEs

# #c_values_adapt= np.random.uniform(1.10,1.30,10) #create ten new trajectories with new values of c
# #chosen from a uniform random dist
# #c_values_adapt=[1.21469163,1.1709559,1.24880952,1.2152118,1.11996975,1.1647565,
# #1.10925619, 1.15028617, 1.16462289, 1.24691801]
# #print(c_values_adapt)

# pers_c_true_array=[]
# pers_c_array=[]
# MSE_values=[]
# RSE_values=[]
# pers_c_difference_array=[]
# count=0

# for value in c_values_adapt:
#             La_o=y_init_values[0]
#             params=[p_l,K_A,d_k,value]
#             pars=[(p_l-d_k), 0, -p_l/K_A, 0, -value]
#             results=odeint(logistic_growth_controlled,La_o,t,args=(params,treatment_function,cycles)) #generate trajectories

#             results= np.array(results)
#             W=np.c_[results,control_values]

#             #separate into y-values and control values
#             y_values=W[:,0] #y values eval at time points
#             controls=W[:,1] #control values eval at time points

#             #estimate the derivative using pySindy
#             fd=FiniteDifference()
#             W_prime=fd._differentiate(results,t)



# #Note that W'=theta(Eps_p+Eps_c) -> W'-theta*Eps_c=theta*Eps_p
# #Now we : 1) Find theta*Eps_C (which are polynomials of x[i] and y[i] with params equal to the common comps) #This is the ESTIMATED derivative
# #         2) let C=W'-theta*Eps_c
# #         3) Solve the problem C=Theta*Eps_p for Eps_p using sparse regression

# #set all the parameters to the common component




# #compute theta*eps_c
# #here, I'm evaluating the recovered model with parameters equal to the common components
# #I'm assuming only c is personalizable, so I evaluate c at its common component and the other parameters at their true values
# #the other parameters should have a personalized comp of 0

#             #Estimate the RHS using the common component
#             Theta_times_eps_c=pars[0]*y_values + pars[1]*controls+ (pars[2])*y_values*y_values + pars[3]*controls*controls-recovered_common*y_values*controls
#             #print("Theta*eps_c", Theta_times_eps_c.shape)
#             Theta_times_eps_c=np.c_[Theta_times_eps_c]
#             #print("Theta*eps_c", Theta_times_eps_c)
#             C=W_prime-Theta_times_eps_c #Calculate C

#             #print("The size of C is ", C.shape)

#             #build the polynomial library and customize to have functions we want
#             theta_lib_c=PolynomialLibrary(degree=2) #restrict to polynomials of deg 2
#             functions = [lambda x : x, lambda x : x*x, lambda x ,y : x*y]
#             customlib=CustomLibrary(library_functions=functions)

#             customlib.fit(W) #fit to our data
#             theta_c=customlib.transform(W) #transform to the correct size (,5)

#             Eps_P=np.linalg.lstsq(theta_c,C)[0] #Is our initial guess using Python's least squares solver
#             Eps_P_row,Eps_P_col=Eps_P.shape


#             #Loop for sparsification
#             Lambda=0.00000001 #Sparsification knob (this was the value given in the PNAS paper for Lorenz)
#             #This loop is adapted from the Sindy_PNAS paper, SI Appendix and converted into Python
#             for k in range(1000):
#                 smallinds=(np.absolute(Eps_P)<Lambda) #find all "very small" coefficients
#             #print(smallinds)
#                 Eps_P[smallinds]=0 #set them=0
#             #print(Eps_P)
#                 for ind in range(Eps_P_col): #number of columns
#                     biginds=np.logical_not(smallinds[:,ind]) #regress dynamics onto remaining terms
#                     Eps_P[biginds,ind]=np.linalg.lstsq(theta_c[:,biginds],C[:,ind])[0]

#             pers_00=Eps_P[0,0]
#             pers_01=Eps_P[1,0]
#             pers_02=Eps_P[2,0]
#             pers_03=Eps_P[3,0]
#             pers_c=-Eps_P[4,0]


# #calculate the parameters for our estimated trajectory X_hat. Each parameter is the estimated common + est. pers.
#             c_est=recovered_common+pers_c

#             pars=[(p_l-d_k), 0, (-p_l/K_A), 0, -c_est]

#             results_hat=odeint(recovered_model,La_o,t,args=(pars,treatment_function,cycles))


#             MSE = np.square(np.subtract(results,results_hat)).mean()
#             RSE=MSE/np.linalg.norm(results[:,0])
#             print("The MSE for trajectory ", count, "is", MSE)
#             print("The RSE for trajectory ", count, "is", RSE)
#             print()

#             MSE_values=np.append(MSE_values,MSE)
#             RSE_values=np.append(RSE_values,RSE)

#             pers_c_true=value-true_c_common
#             pers_c_true_array=np.append(pers_c_true_array,pers_c_true)
#             pers_c_array=np.append(pers_c_array,pers_c)



#             pers_c_true_difference=pers_c_true-pers_c
#             pers_c_difference_array=np.append(pers_c_difference_array,pers_c_true_difference)
#             count+=1

# print("True personalized comp", pers_c_true_array)
# print("Est personalized comp", pers_c_array)
# print("Difference in personalized components", pers_c_difference_array)



# MSE_average=np.average(MSE_values)
# RSE_average=np.average(RSE_values)
# print("The average MSE value is " , MSE_average)
# print("The average RSE value is " , RSE_average)